In [8]:
from langgraph.checkpoint.sqlite import SqliteSaver
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
import os
import sqlite3
import requests

## Create Database SQLlite

In [9]:
import sqlite3

con = sqlite3.connect("/Users/vatsal/Machine Learning/Gen AI/AIproject/loan_data.db")
cur = con.cursor()

# --- schema ---
cur.execute("""
CREATE TABLE customers (
    customer_id     TEXT PRIMARY KEY,
    name            TEXT,
    monthly_income  REAL,
    employment_type TEXT
)
""")

cur.execute("""
CREATE TABLE loans (
    loan_id                TEXT PRIMARY KEY,
    customer_id            TEXT,
    product_type           TEXT,      
    original_amount        REAL,
    outstanding_balance    REAL,
    interest_rate          REAL,      
    rate_type              TEXT,   
    start_date             TEXT,     
    term_months            INTEGER,
    remaining_term_months  INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cur.execute("""
CREATE TABLE repayments (
    repayment_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    loan_id       TEXT,
    due_date      TEXT,       
    amount        REAL,
    status        TEXT,      
    FOREIGN KEY (loan_id) REFERENCES loans(loan_id)
)
""")

# --- seed: 5 customers, each a distinct scenario ---
customers = [
    ('C001', 'Aoife Byrne',    5000, 'full_time'),   # standard fixed
    ('C002', 'Liam Murphy',    6200, 'full_time'),    # variable rate
    ('C003', 'Saoirse Kelly',  3800, 'full_time'),    # has missed payments
    ('C004', 'Cian Walsh',     7500, 'self_employed'),# near end of term
    ('C005', 'Niamh Doyle',    4400, 'full_time'),    # recent loan, early in term
]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

loans = [
    # loan_id, cust, product,             orig,    outstanding, rate,  rate_type, start,        term, remaining
    ('L001','C001','fixed_mortgage',      250000,  240000,      0.035,'fixed',   '2023-01-15', 300,  264),
    ('L002','C002','variable_mortgage',   300000,  180000,      0.045,'variable','2016-06-01', 300,  120),
    ('L003','C003','fixed_mortgage',      200000,  190000,      0.040,'fixed',   '2024-03-10', 300,  282),
    ('L004','C004','fixed_mortgage',      150000,  12000,       0.030,'fixed',   '2005-09-01', 300,  18),
    ('L005','C005','variable_mortgage',   280000,  278000,      0.042,'variable','2026-01-05', 360,  356),
]
cur.executemany("INSERT INTO loans VALUES (?,?,?,?,?,?,?,?,?,?)", loans)

# --- repayments: a few per loan, covering paid / missed / upcoming ---
repayments = [
    ('L001','2026-06-30', 1240, 'paid'),
    ('L001','2026-07-30', 1240, 'paid'),
    ('L001','2026-08-30', 1240, 'upcoming'),
    ('L003','2026-05-30', 1010, 'paid'),
    ('L003','2026-06-30', 1010, 'missed'),   # <- the missed-payment scenario
    ('L003','2026-07-30', 1010, 'missed'),
    ('L003','2026-08-30', 1010, 'upcoming'),
    ('L004','2026-07-30',  700, 'paid'),
    ('L004','2026-08-30',  700, 'upcoming'), # near end of term
]
cur.executemany(
    "INSERT INTO repayments (loan_id, due_date, amount, status) VALUES (?,?,?,?)",
    repayments
)

con.commit()
con.close()
print("Database built.")

Database built.


## Import Data and store in Chroma DB

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from pypdf import PdfReader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob
import os
from dotenv import load_dotenv
load_dotenv()

/var/folders/zk/l_pq96ms1qvfyjkxyzyz0qgm0000gn/T/ipykernel_72661/1444474076.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/vatsal/Machine Learning/Gen AI/AIproject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from langchain_chroma import Chroma

In [3]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [4]:
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10173.88it/s]


In [5]:
pdf_dir = "/Users/vatsal/Machine Learning/Gen AI/AIproject/Loan_docs"

all_chunks=[]

pdf_files = glob.glob("%s/*.pdf" % pdf_dir)
for file in pdf_files:
    filename = os.path.basename(file)
    loan_id = filename.split("_")[0]
    
    loader = PdfReader(file)
    
    docs = [
    Document(
        page_content=page.extract_text() or "",
        metadata={"source": filename,
                    "loan_id":loan_id,
                     "page": i, },
    )
    for i, page in enumerate(loader.pages)
        ]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=300,
        add_start_index=True
        ).split_documents(docs)

    all_chunks.extend(text_splitter)

vectorstore = Chroma.from_documents(documents=all_chunks, 
                        embedding=embeddings,
                        persist_directory="/Users/vatsal/Machine Learning/Gen AI/AIproject/chroma_db",
                        )


print(f"Indexed {len(all_chunks)} chunks from {len(pdf_files)} documents")  



    

    

Indexed 30 chunks from 5 documents


In [6]:
# should return ONLY L003's chunks, never L001/L004 overpayment clauses
results = vectorstore.similarity_search(
    "overpayment penalty",
    k=3,
    filter={"loan_id": "L003"},
)
for r in results:
    print(r.metadata["loan_id"], "-", r.page_content[:80])

L003 - 1.2 Outstanding Balance & Duration: The present outstanding principal balance is
L003 - made by the Borrower during the fixed-rate period shall incur a mandatory financ
L003 - CLAUSE 7. MISSED PAYMENTS, DEFAULT & ARREARS ENFORCEMENT
7.1 Grace Period & Late


In [7]:
print(len(all_chunks))
print(set(c.metadata["loan_id"] for c in all_chunks))

30
{'L001', 'L003', 'L005', 'L002', 'L004'}
